# 03 – Binary monthly snow product (seasonal GLMM)

Converts the monthly **maximum NDSI** into snow / no snow / cloud with a generalized linear mixed model (GLMM), one model per season.

**Model** (fitted once in R with `lme4` by `02_fit_snow_GLMM.R`, coefficients in `data/model/glmm_coefficients_seasons.csv`):
```
ThrX ~ NDSI + (NDSI|Stations) + (NDSI|AltC) + (NDSI|Landcover)
```
with ThrX = snow depth ≥ 1 cm (DJF), 5 cm (MAM), 15 cm (JJA), 15 cm (SON).

**Prediction** (as `re.form = ~(NDSI|AltC)+(NDSI|Landcover)`, i.e. without the station effect):

p = 1 / (1 + exp(−[(b0 + u_AltC + u_LC) + (b1 + v_AltC + v_LC) × NDSI_max]))

**Output, one file per month** `Snow_<year>_<month>.tif`:
- 1 = snow (p ≥ cutoff of the season, the cutoff that maximizes the TSS);
- 0 = no snow;
- 2 = cloud / no data (no NDSI);
- 255 = lake or outside Switzerland.

**Statistics per month:** probability min / median / mean / max, number and % of snow, no snow and cloud pixels. They are written to an Excel file.

**Differences with the original R scripts:**
- The prediction is a numpy calculation on the whole of Switzerland, a few seconds per month: no need to split the country in two halves or to build seasonal stacks. The season is given by the month.
- November 2020: 98.9 % of the pixels are identical to the original R map. The difference comes from the grids (6200 rows in the R maps vs. 6199 for the NDSI).

In [ ]:
import os, glob, time
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Configuration section

# monthly NDSI products (band 'max' is used); several folders possible, e.g. old datacube products + STAC products
ndsi_dirs = ['../outputs/NDSI_monthly_STAC']           # add here the folder of the 1984-2021 products, e.g. '/path/to/NDSI_datacube'
ndsi_pattern = 'NDSI_%i_%02d*.tif'            # file name of a month (year, month)

coefs_csv = '../data/model/glmm_coefficients_seasons.csv'     # from 02_fit_snow_GLMM.R
classes_tif = '../data/sources/landcover_elevation_classes.tif'                 # band 1 = Landcover (1,2,3), band 2 = AltC (1-6)
mask_all_tif = '../data/sources/mask_switzerland_without_lakes.tif'                          # 1 = Switzerland without lakes, NA elsewhere

out_dir = '../outputs/Snow_monthly_GLMM'
aux_dir = '../data/aux'
stats_xlsx = '%s/Snow_monthly_GLMM_stats.xlsx' % out_dir
save_proba = False                            # True: also write the probability (Proba_<year>_<month>.tif)

start_year, end_year = 1984, 2026
start_month, end_month = 1, 12

# season of each month
season_of_month = {12: 'DJF', 1: 'DJF', 2: 'DJF', 3: 'MAM', 4: 'MAM', 5: 'MAM',
                   6: 'JJA', 7: 'JJA', 8: 'JJA', 9: 'SON', 10: 'SON', 11: 'SON'}

# NDSI grid (same as the NDSI products of the datacube and STAC versions)
grid_origin = (5.699957268017272, 47.85026882419969)
grid_res    = (0.0003407435000010325, 0.00034046861818428666)
grid_size   = (14234, 6199)                    # (columns, rows)
grid_transform = from_origin(grid_origin[0], grid_origin[1], grid_res[0], grid_res[1])
grid_crs = 'EPSG:4326'
W, H = grid_size

os.makedirs(out_dir, exist_ok=True)
os.makedirs(aux_dir, exist_ok=True)

## 1. Auxiliary layers on the NDSI grid (created once)

The land cover / elevation classes and the Switzerland mask are on a grid of 14234 × 6200 pixels, very close to the NDSI grid. They are resampled once onto the NDSI grid with nearest neighbour, since they are classes.
- Land cover and AltC are coded as uint8 (0 = NA).
- The mask is 1 in Switzerland without lakes, 0 elsewhere.

In [ ]:
def to_grid(src_path, band, out_path):
    """Resample one band (classes) onto the NDSI grid (nearest), uint8 with 0 = NA."""
    if not os.path.exists(out_path):
        with rasterio.open(src_path) as src:
            a = np.zeros((H, W), 'float32')
            reproject(rasterio.band(src, band), a, src_transform=src.transform, src_crs=src.crs,
                      dst_transform=grid_transform, dst_crs=grid_crs,
                      src_nodata=src.nodata, dst_nodata=0, resampling=Resampling.nearest)
        a[~np.isfinite(a) | (np.abs(a) > 100)] = 0
        prof = dict(driver='GTiff', width=W, height=H, count=1, dtype='uint8', crs=grid_crs,
                    transform=grid_transform, nodata=0, compress='deflate')
        with rasterio.open(out_path, 'w', **prof) as dst:
            dst.write(a.astype('uint8'), 1)
    with rasterio.open(out_path) as f:
        return f.read(1)

LC   = to_grid(classes_tif, 1, '%s/landcover.tif' % aux_dir)
ALTC = to_grid(classes_tif, 2, '%s/elevation_classes.tif' % aux_dir)
MASK = to_grid(mask_all_tif, 1, '%s/mask_switzerland_without_lakes.tif' % aux_dir) == 1

print('Landcover classes', np.unique(LC), '| AltC classes', np.unique(ALTC), '| pixels in Switzerland', MASK.sum())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 4))
for a, (arr, t) in zip(ax, [(LC, 'Landcover'), (ALTC, 'AltC'), (MASK, 'Switzerland without lakes')]):
    a.imshow(arr[::5, ::5], interpolation='nearest'); a.set_title(t); a.axis('off')

## 2. Model coefficients (R, lme4)

For each season, an intercept and a slope are computed for each combination AltC × Landcover:
- intercept = b0 + u_AltC + u_LC;
- slope = b1 + v_AltC + v_LC.

The prediction of a pixel is then a simple lookup in this table.

In [ ]:
coefs = pd.read_csv(coefs_csv)
display(coefs[coefs.effect == 'fixed'][['season', 'snow_thr_cm', 'intercept', 'slope', 'cutoff', 'tss']])

# lookup tables [AltC, Landcover] (index 0 = NA)
models = {}
for s in ['DJF', 'MAM', 'JJA', 'SON']:
    c = coefs[coefs.season == s]
    fx = c[c.effect == 'fixed'].iloc[0]
    I = np.full((7, 4), np.nan, 'float32'); S = np.full((7, 4), np.nan, 'float32')
    alt = c[c.effect == 'AltC'].set_index('level'); lc = c[c.effect == 'Landcover'].set_index('level')
    for a in range(1, 7):
        for l in range(1, 4):
            I[a, l] = fx.intercept + alt.loc[a, 'intercept'] + lc.loc[l, 'intercept']
            S[a, l] = fx.slope + alt.loc[a, 'slope'] + lc.loc[l, 'slope']
    models[s] = dict(I=I, S=S, cutoff=float(fx.cutoff), snow_thr_cm=int(fx.snow_thr_cm))
    print('%s: snow >= %i cm, cutoff %.2f' % (s, models[s]['snow_thr_cm'], models[s]['cutoff']))

## 3. Functions

In [ ]:
def find_ndsi(year, month):
    """Monthly NDSI file of a month (first found in ndsi_dirs), or None."""
    for d in ndsi_dirs:
        f = sorted(glob.glob(os.path.join(d, ndsi_pattern % (year, month))))
        if f:
            return f[0]
    return None


def read_ndsi_max(path):
    """Band 'max' of a monthly NDSI product on the NDSI grid (reprojected if needed), NaN = no data."""
    with rasterio.open(path) as src:
        b = list(src.descriptions).index('max') + 1 if 'max' in src.descriptions else 1
        same = (src.width, src.height) == (W, H) and src.transform.almost_equals(grid_transform, 1e-9)
        if same:
            a = src.read(b).astype('float32')
            if src.nodata is not None:
                a[a == src.nodata] = np.nan
        else:
            a = np.full((H, W), np.nan, 'float32')
            reproject(rasterio.band(src, b), a, src_transform=src.transform, src_crs=src.crs,
                      dst_transform=grid_transform, dst_crs=grid_crs, src_nodata=src.nodata,
                      dst_nodata=np.nan, resampling=Resampling.nearest)
    a[(a > 1) | (a < -1)] = np.nan          # as in the R script
    return a


def predict_snow(ndsi, season):
    """Probability of snow (GLMM of the season) and binary product 0/1/2/255."""
    m = models[season]
    p = 1 / (1 + np.exp(-(m['I'][ALTC, LC] + m['S'][ALTC, LC] * ndsi)))   # NaN if NDSI, AltC or LC is NA
    snow = np.full((H, W), 255, 'uint8')                     # 255 = NA (lakes, outside Switzerland)
    snow[MASK] = 2                                           # 2 = cloud / no data
    ok = MASK & np.isfinite(p)
    snow[ok] = (p[ok] >= m['cutoff']).astype('uint8')       # 1 = snow, 0 = no snow
    return p, snow


def write_tif(path, arr, dtype, nodata):
    prof = dict(driver='GTiff', width=W, height=H, count=1, dtype=dtype, crs=grid_crs, transform=grid_transform,
                nodata=nodata, compress='deflate', tiled=True)
    with rasterio.open(path, 'w', **prof) as dst:
        dst.write(arr.astype(dtype), 1)


def month_stats(year, month, season, p, snow):
    """Statistics of the month (as in the original R scripts)."""
    pv = p[MASK & np.isfinite(p)]
    n_ch = int(MASK.sum())
    n1, n0, n2 = [int((snow == k).sum()) for k in (1, 0, 2)]
    return dict(Date='%i-%02d' % (year, month), Season=season,
                Min=float(pv.min()) if pv.size else np.nan, Median=float(np.median(pv)) if pv.size else np.nan,
                Mean=float(pv.mean()) if pv.size else np.nan, Max=float(pv.max()) if pv.size else np.nan,
                snow=n1, no_snow=n0, cloud=n2, NoData=int(H * W - n_ch),
                snow_pct=100 * n1 / n_ch, no_snow_pct=100 * n0 / n_ch, cloud_pct=100 * n2 / n_ch)

## 4. Test on one month

The first available NDSI month. (The comparison with the original R maps, 98.9 % identical pixels in November 2020, is described in the README.)

In [ ]:
# test on one month (the first NDSI product found)
test = None
for y in range(start_year, end_year + 1):
    for m in range(1, 13):
        if test is None and find_ndsi(y, m):
            test = (y, m)
t0 = time.time()
ndsi = read_ndsi_max(find_ndsi(*test))
p, snow = predict_snow(ndsi, season_of_month[test[1]])
print('%i-%02d done in %.0f s' % (test[0], test[1], time.time() - t0)); month_stats(test[0], test[1], season_of_month[test[1]], p, snow)

## 5. All months

- Months already computed are skipped.
- A month without NDSI file is ignored.
- The statistics are added to the Excel file after each month.

In [ ]:
stats = pd.read_excel(stats_xlsx).to_dict('records') if os.path.exists(stats_xlsx) else []
done = {r['Date'] for r in stats}

for year in range(start_year, end_year + 1):
    for month in range(start_month, end_month + 1):
        out_tif = '%s/Snow_%i_%02d.tif' % (out_dir, year, month)
        if os.path.exists(out_tif):
            continue
        f = find_ndsi(year, month)
        if f is None:
            continue
        t0 = time.time()
        season = season_of_month[month]
        p, snow = predict_snow(read_ndsi_max(f), season)
        write_tif(out_tif, snow, 'uint8', 255)
        if save_proba:
            write_tif('%s/Proba_%i_%02d.tif' % (out_dir, year, month), np.where(MASK, p, np.nan), 'float32', np.nan)
        st = month_stats(year, month, season, p, snow)
        stats = [r for r in stats if r['Date'] != st['Date']] + [st]
        pd.DataFrame(stats).sort_values('Date').to_excel(stats_xlsx, index=False)
        print('%i-%02d %s: snow %.1f %% | no snow %.1f %% | cloud %.1f %% (%.0f s)'
              % (year, month, season, st['snow_pct'], st['no_snow_pct'], st['cloud_pct'], time.time() - t0))

print('ALL DONE')

In [ ]:
df_stats = pd.read_excel(stats_xlsx)
df_stats.tail(12)